In [2]:
import os
import pickle
# import evaluate 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm


## Fetch narratives

In [12]:
# list of files
narr_files = [
    "gpt-4o-mini_0.0_given_local.json",
    "gpt-4o-mini_0.5_given_local.json",
    "gpt-4o-mini_1.2_given_local.json"
]
narr_dir = "data/narratives/"

def load_json_narratives_to_df(narr_dir, narr_files):
    df_list = [
        format_narrative_df(pd.read_json(os.path.join(narr_dir,file)), file)
        for file in tqdm(narr_files, desc="Loading narrative file")
    ]
    narr_df = pd.concat(df_list, ignore_index=True)

    return narr_df

def format_narrative_df(df, file_name):
    print(f"Formatting: {file_name}")
    df_formatted = df.melt(var_name='profile_id', value_name='text')
    df_formatted['model'] = file_name.split(sep='_')[0]
    df_formatted['temperature'] = float(file_name.split(sep='_')[1])
    df_formatted['scenario'] = file_name.split(sep='_')[2]
    df_formatted['sample'] = df_formatted.groupby('profile_id').cumcount() + 1

    return df_formatted

narrs = load_json_narratives_to_df(narr_dir, narr_files)

Loading narrative file:   0%|          | 0/3 [00:00<?, ?it/s]

Formatting: gpt-4o-mini_0.0_given_local.json


Loading narrative file:  67%|██████▋   | 2/3 [00:03<00:01,  1.70s/it]

Formatting: gpt-4o-mini_0.5_given_local.json


Loading narrative file: 100%|██████████| 3/3 [00:05<00:00,  1.71s/it]

Formatting: gpt-4o-mini_1.2_given_local.json


Loading narrative file: 100%|██████████| 3/3 [00:05<00:00,  1.75s/it]


In [15]:
narrs

,profile_id,text,model,temperature,scenario,sample
0,1,"As the sun began to rise over the barracks, Ma...",gpt-4o-mini,0.0,given,1
1,1,"As the sun began to rise over the barracks, Ma...",gpt-4o-mini,0.0,given,2
2,1,"As the sun began to rise over the barracks, Ma...",gpt-4o-mini,0.0,given,3
3,1,"As the sun began to rise over the barracks, Pr...",gpt-4o-mini,0.0,given,4
4,1,"As the sun began to rise over the barracks, Ma...",gpt-4o-mini,0.0,given,5
...,...,...,...,...,...,...
63355,1056,Samantha Caldwell stared out the window of her...,gpt-4o-mini,1.2,given,16
63356,1056,The sun slipped behind the horizon as Ava Morg...,gpt-4o-mini,1.2,given,17
63357,1056,Standing on the uneven wooden deck of her mode...,gpt-4o-mini,1.2,given,18
63358,1056,"At the edge of dawn, Clara stands by her kitch...",gpt-4o-mini,1.2,given,19


In [16]:
male_profiles = [i+1 for i in range(528)]
female_profiles = [i+528 for i in male_profiles]

male_narrs = narrs[narrs['profile_id'].isin(male_profiles)]['text'].tolist()
female_narrs = narrs[narrs['profile_id'].isin(female_profiles)]['text'].tolist()

## Preprocessing

In [4]:
# let's see if it's needed

## Regard

In [ ]:
regard = evaluate.load("regard", "compare")

Using the latest cached version of the module from C:\Users\Nadia Timoleon\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-measurement--regard\49c8ca499f140affc8972ee0478a52401e4537b1ebde0d486418fea1d4504625 (last modified on Mon Apr  7 18:36:34 2025) since it couldn't be found locally at evaluate-measurement--regard, or remotely on the Hugging Face Hub.
Device set to use cpu


In [27]:
a = 0
b = 100

In [29]:
regard.compute(data=male_narrs[a:b], references=female_narrs[a:b])

{'regard_difference': {'other': 0.013514597117900817,
  'positive': -0.05705872565507886,
  'negative': 0.03348317246884108,
  'neutral': 0.01006095975637436}}

## Sentiment

### SiEBERT

In [8]:
from transformers import pipeline

In [9]:
sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")

Device set to use cpu


In [17]:
print(sentiment_analysis(male_narrs[0]))

[{'label': 'POSITIVE', 'score': 0.9982195496559143}]


In [18]:
print(sentiment_analysis(female_narrs[0]))

[{'label': 'POSITIVE', 'score': 0.996405839920044}]


In [ ]:
df = narrs.copy()

# Batch processing
batch_size = 500  # Adjust based on your system
result_list = []

start_batch = 1
start_index = start_batch * batch_size

# Load previously saved batches into result_list
# for b in range(1, start_batch):
#     with open(f'data/analysis/narrs_siebert_temp_batch_{b}.pkl', 'rb') as f:
#         result_list.append(pickle.load(f))

for i in range(start_index, len(df), batch_size):
    tqdm.pandas()
    batch = df.iloc[i:i+batch_size].copy()
    current_batch_size = i//batch_size + 1
    print(f"Processing batch {current_batch_size}...")
    
    batch['siebert'] = batch['text'].progress_apply(sentiment_analysis)
    result_list.append(batch)

    # save intermediate batch
    with open(f'data/analysis/narrs_siebert_temp_batch_{current_batch_size}.pkl', 'wb') as f:
        pickle.dump(batch, f)

# Combine all processed batches
final_df = pd.concat(result_list, ignore_index=True)

# Save final DataFrame
with open('data/analysis/narrs_siebert.pkl', 'wb') as f:
    pickle.dump(final_df, f)

Processing batch 2...


  4%|▍         | 76/2000 [03:33<1:30:12,  2.81s/it]


KeyboardInterrupt: 

### tabularisai/robust-sentiment-analysis

In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [10]:
# Load model and tokenizer
model_name = "tabularisai/robust-sentiment-analysis"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Function to predict sentiment
def predict_sentiment(text):
    inputs = tokenizer(text.lower(), return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    
    sentiment_map = {0: "Very Negative", 1: "Negative", 2: "Neutral", 3: "Positive", 4: "Very Positive"}
    return sentiment_map[predicted_class]

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/819 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
predict_sentiment(male_narrs[0])

'Neutral'

In [13]:
predict_sentiment(female_narrs[0])

'Neutral'

In [23]:
df = narrs.copy()

# Batch processing
batch_size = 2000  # Adjust based on your system
result_list = []

start_batch = 18
start_index = start_batch * batch_size

# Load previously saved batches into result_list
for b in range(1, start_batch):
    with open(f'data/analysis/narrs_robust_sentiment_temp_batch_{b}.pkl', 'rb') as f:
        result_list.append(pickle.load(f))

for i in range(start_index, len(df), batch_size):
    tqdm.pandas()
    batch = df.iloc[i:i+batch_size].copy()
    current_batch_size = i//batch_size + 1
    print(f"Processing batch {current_batch_size}...")
    
    batch['robust_sentiment_analysis'] = batch['text'].progress_apply(predict_sentiment)
    result_list.append(batch)

    # save intermediate batch
    with open(f'data/analysis/narrs_robust_sentiment_temp_batch_{current_batch_size}.pkl', 'wb') as f:
        pickle.dump(batch, f)

# Combine all processed batches
final_df = pd.concat(result_list, ignore_index=True)

# Save final DataFrame
with open('data/analysis/narrs_robust_sentiment.pkl', 'wb') as f:
    pickle.dump(final_df, f)

Processing batch 19...


100%|██████████| 2000/2000 [08:32<00:00,  3.90it/s]


Processing batch 20...


100%|██████████| 2000/2000 [08:42<00:00,  3.82it/s]


Processing batch 21...


...

100%|██████████| 2000/2000 [09:25<00:00,  3.54it/s]


Processing batch 31...


100%|██████████| 2000/2000 [09:23<00:00,  3.55it/s]


Processing batch 32...


100%|██████████| 1360/1360 [06:24<00:00,  3.54it/s]


In [26]:
with open(f'data/analysis/narrs_robust_sentiment.pkl', 'rb') as f:
  narrs_robust_sentiment = pickle.load(f)

In [27]:
narrs_robust_sentiment.head(5)

,profile_id,text,model,temperature,scenario,sample,robust_sentiment_analysis
0,1,"As the sun began to rise over the barracks, Ma...",gpt-4o-mini,0.0,given,1,Neutral
1,1,"As the sun began to rise over the barracks, Ma...",gpt-4o-mini,0.0,given,2,Neutral
2,1,"As the sun began to rise over the barracks, Ma...",gpt-4o-mini,0.0,given,3,Neutral
3,1,"As the sun began to rise over the barracks, Pr...",gpt-4o-mini,0.0,given,4,Neutral
4,1,"As the sun began to rise over the barracks, Ma...",gpt-4o-mini,0.0,given,5,Neutral


In [28]:
male_profiles = [i+1 for i in range(528)]
female_profiles = [i+528 for i in male_profiles]

male_narrs = narrs_robust_sentiment[narrs_robust_sentiment['profile_id'].isin(male_profiles)]
female_narrs = narrs_robust_sentiment[narrs_robust_sentiment['profile_id'].isin(female_profiles)]

In [35]:
male_narrs['robust_sentiment_analysis'].value_counts()

robust_sentiment_analysis
Neutral          28344
Positive          3024
Very Positive      168
Negative           130
Very Negative       14
Name: count, dtype: int64

In [37]:
male_narrs[male_narrs['robust_sentiment_analysis']=='Very Negative']

,profile_id,text,model,temperature,scenario,sample,robust_sentiment_analysis
21997,44,"Jamal sat on the worn-out couch in his small, ...",gpt-4o-mini,0.5,given,18,Very Negative
23743,132,Carlos sat on the worn-out couch in his small ...,gpt-4o-mini,0.5,given,4,Very Negative
23757,132,Carlos sat on the worn-out couch in his small ...,gpt-4o-mini,0.5,given,18,Very Negative
28141,352,The sun spilled through the worn curtains of J...,gpt-4o-mini,0.5,given,2,Very Negative
30943,492,As the sun peeked through the window of his sm...,gpt-4o-mini,0.5,given,4,Very Negative
30998,494,As the morning light streamed through the wind...,gpt-4o-mini,0.5,given,19,Very Negative
41108,44,The sun peeked through the peeling curtains of...,gpt-4o-mini,1.2,given,9,Very Negative
42259,101,Carlos sat at his kitchen table in a cramped a...,gpt-4o-mini,1.2,given,20,Very Negative
42958,136,"At fifty-two, Samuel Harris could never have a...",gpt-4o-mini,1.2,given,19,Very Negative
43104,144,"Martin glanced at the screen, its dull glow re...",gpt-4o-mini,1.2,given,5,Very Negative


In [40]:
print(narrs.iloc[42958]['text'])

Carlos stood in the assembly line of a local furniture factory, his hands methodically fitting wooden legs to tabletops, the rhythmic clatter of tools a backdrop to his daily routine. At 32, education had never been a priority in his impoverished childhood. His father, a factory worker like him, had urged Carlos to find stability over ambition, passing down the echoes of lost dreams. 

Single, Carlos often found comfort in quiet evenings; a second-hand guitar propped against the wall served as a silent friend, its strings vibrating with the hopes he held tightly within. Music was his escape—a world far removed from the dust and sweat of the factory, allowing him to dream of stages instead of assembly lines.

Yet, as his coworkers spoke of plans to settle down or aspirations to climb the ranks, Carlos felt a nagging sense of longing. He wanted more, not just for himself, but to be a role model for his younger sister, who was striving to break free from the same cycle of unfulfilled pote

In [36]:
female_narrs['robust_sentiment_analysis'].value_counts()

robust_sentiment_analysis
Neutral          24526
Positive          4356
Very Positive      746
Negative            51
Very Negative        1
Name: count, dtype: int64